# Group Convolution — Companion Notebook

**6.7970/8.750 Symmetry and its Application to Machine Learning**

This notebook follows the Group Convolution exercise section by section.
Use it to **prototype your code** and **test your implementations**
against the course library before submitting on the website.

Each section includes small tests you can use to check your work.

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/atomicarchitects/symm4ml-colabs/blob/main/group_conv_companion.ipynb)

## Setup

In [ ]:
%%capture
!pip install torch
!pip install https://symm4ml.mit.edu/_static/symm4ml_s26/symm4ml/symm4ml_latest.zip

In [ ]:
import numpy as np
import itertools
import torch
import torch.nn.functional as F

from symm4ml import groups, rep, vib_modes, grids, group_conv

### Reference data

These groups, representations, and test tensors are used throughout the exercise for testing.

In [ ]:
# C4 group (cyclic group of order 4, 2D rotation matrices)
_subset_C4 = np.array([
    [[np.cos(np.pi / 2), -np.sin(np.pi / 2)],
     [np.sin(np.pi / 2),  np.cos(np.pi / 2)]],
])
C4 = groups.generate_group(_subset_C4)
C4 = C4[[3, 1, 0, 2]]  # reorder so E is first
C4_torch = torch.tensor(C4).to(torch.float32)
C4_table = groups.make_multiplication_table(C4)
C4_reg_rep_torch = torch.tensor(rep.regular_representation(C4_table)).to(torch.float32)

# D4 group (dihedral group of order 8, 2D rotation + mirror matrices)
_subset_D4 = np.array([
    [[-1., 0.], [0., 1.]],
    [[np.cos(np.pi / 2), -np.sin(np.pi / 2)],
     [np.sin(np.pi / 2),  np.cos(np.pi / 2)]],
])
D4 = groups.generate_group(_subset_D4)
D4 = np.array(D4[::-1])  # reorder so E is first
D4_torch = torch.tensor(D4).to(torch.float32)
D4_table = groups.make_multiplication_table(D4)
D4_reg_rep_torch = torch.tensor(rep.regular_representation(D4_table)).to(torch.float32)

# Test tensors for reg_rep_group_convolution
trial1_input = torch.arange(0, 4, dtype=torch.float32).reshape(1, 1, 4)
trial1_filter = torch.arange(4, 8, dtype=torch.float32).reshape(1, 1, 4)
trial2_input = torch.arange(3 * 8, dtype=torch.float32).reshape(1, 3, 8)
trial2_filter = torch.arange(2 * 3 * 8, dtype=torch.float32).reshape(2, 3, 8)

# Test tensors for image2D functions
test_image_1 = torch.arange(4 * len(D4) * 5 * 5, dtype=torch.float32).reshape(1, 4, len(D4), 5, 5) * 0.1
test_filter_1 = torch.arange(2 * 4 * len(D4) * 3 * 3, dtype=torch.float32).reshape(2, 4, len(D4), 3, 3) * 0.1
test_image_2 = torch.arange(2 * len(C4) * 7 * 7, dtype=torch.float32).reshape(2, 1, len(C4), 7, 7) * 0.1
test_filter_2 = torch.arange(4 * len(C4) * 3 * 3, dtype=torch.float32).reshape(4, 1, len(C4), 3, 3) * 0.1

print(f"C4: {len(C4)} elements")
print(f"D4: {len(D4)} elements")

---
## 1. `reg_rep_group_convolution(reg_rep, input, filter)`

Group convolution over the left regular representation. Using Einstein summation notation:

$$[f \star \psi]_{zdk} = \sum_{i \in G} \sum_{j \in G} f_{zci}\, \psi_{dcj}\, D^{\text{reg}}_{kij}$$

Implement this with `torch.einsum`.

In [ ]:
def reg_rep_group_convolution(reg_rep, input, filter):
    """Performs group convolution of inputs and filters over the regular representation
    Input:
        reg_rep: torch.Tensor of shape [|G|, |G|, |G|] of the left regular representation
        input: torch.Tensor of shape [batch, channel_in, reg_rep_in]
        filter: torch.Tensor of shape [channel_out, channel_in, reg_rep_filter]
    Output:
        output: torch.Tensor of shape [batch, channel_out, reg_rep_out]
    """
    # YOUR CODE HERE
    pass

In [ ]:
# No small tests in group_conv.py for reg_rep_group_convolution.
# Supplementary comparison with course (not a course small test):
np.testing.assert_allclose(
    reg_rep_group_convolution(C4_reg_rep_torch, trial1_input, trial1_filter).numpy(),
    group_conv.reg_rep_group_convolution(C4_reg_rep_torch, trial1_input, trial1_filter).numpy(),
    atol=1e-5
)
np.testing.assert_allclose(
    reg_rep_group_convolution(D4_reg_rep_torch, trial2_input, trial2_filter).numpy(),
    group_conv.reg_rep_group_convolution(D4_reg_rep_torch, trial2_input, trial2_filter).numpy(),
    atol=1e-5
)
print("reg_rep_group_convolution comparison passed!")

---
## Grid Utilities

Two simple utilities for assigning coordinates and getting indices for tensor/grid entries.

### 2. `grid_coords(dims)`

Create a coordinate grid for a tensor of specified dimensions. Each coordinate is generated by taking a `linspace` from $-1$ to $1$ for each dimension, then forming the Cartesian product.

In [ ]:
def grid_coords(dims):
    """Create a coordinate grid for a tensor of specified dimensions.

    Input:
        dims : (iterable of int) The shape of the tensor. Each element
            specifies the number of pointsalong that dimension.

    Output:
        np.array representing the tensor coordinates. The returned array
        has shape (dof, d), where dof = ∏ dims (the total number of points)
        and d = len(dims). Each coordinate is generated by taking a linspace
        from -1 to 1 for each dimension.

    Notes:
        This function uses itertools.product to construct the full grid
        of coordinates from the provided dimensions.

    Examples:
    >>> grid_coords([2, 3])
    array([-1., -1.],
          [-1.,  0.],
          [-1.,  1.],
          [ 1., -1.],
          [ 1.,  0.],
          [ 1.,  1.]])
    """
    # YOUR CODE HERE
    pass

In [ ]:
# No small tests in grids.py for grid_coords.
# Supplementary comparison with course (not a course small test):
np.testing.assert_allclose(
    grid_coords([2, 3]),
    grids.grid_coords([2, 3]),
    atol=1e-7
)
np.testing.assert_allclose(
    grid_coords([3, 3, 3]),
    grids.grid_coords([3, 3, 3]),
    atol=1e-7
)
print("grid_coords comparison passed!")

### 3. `grid_indices(dims)`

Create an index grid for a tensor of specified dimensions. Each index is generated with `arange` for each dimension, then forming the Cartesian product.

In [ ]:
def grid_indices(dims):
    """Create a coordinate grid for a tensor of specified dimensions.

    Input:
        dims : (iterable of int) The shape of the tensor. Each element 
        specifies the number of points along that dimension.

    Output:
        np.array representing the tensor indices such that they can be 
        flattened. The returned array has shape (dof, d), where dof = ∏ dims
        (the total number of points) and d = len(dims). Each coordinate is 
        generated with arange for each dimension.

    Notes:
        This function uses itertools.product to construct the full grid of
        indices from the provided dimensions.

    Examples:
    >>> grid_indices([2, 3])
    array([[0, 0],
           [0, 1],
           [0, 2],
           [1, 0],
           [1, 1],
           [1, 2]])
    """
    # YOUR CODE HERE
    pass

In [ ]:
# No small tests in grids.py for grid_indices.
# Supplementary comparison with course (not a course small test):
np.testing.assert_array_equal(
    grid_indices([2, 3]),
    grids.grid_indices([2, 3])
)
np.testing.assert_array_equal(
    grid_indices([3, 3, 3]),
    grids.grid_indices([3, 3, 3])
)
print("grid_indices comparison passed!")

---
## 2D Image Group Convolution

For images, group convolution operates over both spatial (2D rotations/mirrors) and regular representations:

$$[f \star \psi]_{zdk}(\mathbf{x}) = \sum_{\mathbf{y}} \sum_{i \in G} \sum_{j \in G} f_{zci}(\mathbf{x} + k\mathbf{y})\,\psi_{dcj}(\mathbf{y})\,D^{\text{reg}}(k)_{ij}$$

We implement this in three steps: compute the permutation representation for kernel pixels, create a rotated filter bank, then perform the full convolution.

### 4. `image2D_permutation_representation(rep_2D, kernel_size)`

Compute the permutation representation for pixels of a 2D image kernel. This bridges NumPy (geometric coordinate computation) and PyTorch (convolution operations).

**Hints:**
- Use `grids.grid_coords` to get pixel coordinates, multiplied by `np.array([1, -1])` to match image conventions (top-left = min x, max y).
- Use `vib_modes.permutation_representation` to compute the permutation representation from coordinates and 2D group matrices.

In [ ]:
def image2D_permutation_representation(rep_2D, kernel_size):
    """Creates the permutation representation for pixels of a 2D image kernel.
    Input:
        rep_2D: torch.Tensor of shape [|G|, 2, 2]
        kernel_size: list of [kernel_height, kernel_width]
    Output:
        perm_rep: torch.Tensor of shape [|G|, kh*kw, kh*kw]
    """
    # YOUR CODE HERE
    pass

In [ ]:
# No small tests in group_conv.py for image2D_permutation_representation.
# Supplementary comparison with course (not a course small test):
np.testing.assert_allclose(
    image2D_permutation_representation(C4_torch, [3, 3]).numpy(),
    group_conv.image2D_permutation_representation(C4_torch, [3, 3]).numpy(),
    atol=1e-5
)
np.testing.assert_allclose(
    image2D_permutation_representation(D4_torch, [3, 3]).numpy(),
    group_conv.image2D_permutation_representation(D4_torch, [3, 3]).numpy(),
    atol=1e-5
)
print("image2D_permutation_representation comparison passed!")

### 5. `image2D_group_convolution_filter_bank(perm_rep, filter)`

Create a rotated filter bank using the permutation representation. The `perm_rep` acts on the flattened spatial dimensions of the filter and adds a new axis spanning group elements. This is a single `torch.einsum` plus a reshape.

In [ ]:
def image2D_group_convolution_filter_bank(perm_rep, filter):
    """Creates rotated filter bank for 2D image convolution
    Input:
        perm_rep: torch.Tensor of shape [|G|, kh*kw, kh*kw]
        filter: torch.Tensor of shape [channel_out, channel_in, reg_rep_filter, kernel_height, kernel_width]
    Output:
        filter_bank: torch.Tensor of shape [reg_rep_out, channel_out, channel_in, reg_rep_filter, kernel_height, kernel_width]
    """
    # YOUR CODE HERE
    pass

In [ ]:
# No small tests in group_conv.py for image2D_group_convolution_filter_bank.
# Supplementary comparison with course (not a course small test):
_D4_perm_rep_3x3 = group_conv.image2D_permutation_representation(D4_torch, [3, 3])
_C4_perm_rep_3x3 = group_conv.image2D_permutation_representation(C4_torch, [3, 3])
np.testing.assert_allclose(
    image2D_group_convolution_filter_bank(_D4_perm_rep_3x3, test_filter_1).numpy(),
    group_conv.image2D_group_convolution_filter_bank(_D4_perm_rep_3x3, test_filter_1).numpy(),
    atol=1e-5
)
np.testing.assert_allclose(
    image2D_group_convolution_filter_bank(_C4_perm_rep_3x3, test_filter_2).numpy(),
    group_conv.image2D_group_convolution_filter_bank(_C4_perm_rep_3x3, test_filter_2).numpy(),
    atol=1e-5
)
print("image2D_group_convolution_filter_bank comparison passed!")

### 6. `image2D_group_convolution(perm_rep, reg_rep, input, filter)`

Put everything together: create the rotated filter bank, contract with the regular representation, then use `F.conv2d` with `padding=1` for the spatial convolution.

**Hints:**
- Call `image2D_group_convolution_filter_bank` to create the rotated filter bank.
- Use `torch.einsum` with the `reg_rep` to contract the regular representation indices.
- Reshape input and filter bank to merge channel/group dimensions for `F.conv2d`, then unflatten the output.

In [ ]:
def image2D_group_convolution(perm_rep, reg_rep, input, filter):
    """Performs group convolution of inputs and filters over the regular representation
    Input:
        perm_rep: torch.Tensor of shape [|G|, kh*kw, kh*kw]
        reg_rep: torch.Tensor of shape [|G|, |G|, |G|] of the left regular representation
        input: torch.Tensor of shape [batch, channel_in, reg_rep_in, height, width]
        filter: torch.Tensor of shape [channel_out, channel_in, reg_rep_filter, kernel_height, kernel_width]
    Output:
        output: torch.Tensor of shape [batch, channel_out, reg_rep_out, height, width]
    """
    # YOUR CODE HERE
    pass

In [ ]:
# No small tests in group_conv.py for image2D_group_convolution.
# Supplementary comparison with course (not a course small test):
_D4_perm_rep_3x3 = group_conv.image2D_permutation_representation(D4_torch, [3, 3])
_C4_perm_rep_3x3 = group_conv.image2D_permutation_representation(C4_torch, [3, 3])
np.testing.assert_allclose(
    image2D_group_convolution(_D4_perm_rep_3x3, D4_reg_rep_torch, test_image_1, test_filter_1).detach().numpy(),
    group_conv.image2D_group_convolution(_D4_perm_rep_3x3, D4_reg_rep_torch, test_image_1, test_filter_1).detach().numpy(),
    atol=1e-3
)
np.testing.assert_allclose(
    image2D_group_convolution(_C4_perm_rep_3x3, C4_reg_rep_torch, test_image_2, test_filter_2).detach().numpy(),
    group_conv.image2D_group_convolution(_C4_perm_rep_3x3, C4_reg_rep_torch, test_image_2, test_filter_2).detach().numpy(),
    atol=1e-3
)
print("image2D_group_convolution comparison passed!")

---
## Explore Further

In [ ]:
# Try a different group or kernel size!

In [ ]:
# Experiment with group equivariance:
# Verify that L_u [f * psi] == [L_u f] * psi for some group element u